# 03 표 확장 + 사진 수집 (Colab, GPU 필요 없음)
1. **표 확장**: 모델이 아는 101종 + 새 음식 후보 74종(`knowledge/candidates.csv`) → 알레르기 비율 + 영양 중앙값 표
2. **사진 수집**: 그 레시피들의 사진 주소(`image1`)를 받아 작게 줄여 드라이브에 저장 → CLIP 검색기 · 처음 보는 음식 감지 학습용
   - 주소는 시간이 지나면 계속 죽어서 **한 번에 모아 둔다**. 몇 %가 살아 있는지도 같이 센다.
   - 사진은 레시피 사이트 것이라 **팀 연구용으로만**, 깃에는 절대 올리지 않는다.

위에서부터 순서대로 실행. 2번은 1시간 안팎이고, 끊겨도 다시 실행하면 이미 받은 음식은 건너뛴다.

## 0. 드라이브 연결 · 코드 받기

In [ ]:
import os, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
CSV = f'{DATA}/1M_Final.csv'
OUT = f"{find(TEAM, '찍먹_실험기록')}/표"          # 표 결과 (깃에 올릴 것)
IMG = f'{DATA}/images'                              # 사진 (재배포 금지, 드라이브에만)
os.makedirs(OUT, exist_ok=True); os.makedirs(IMG, exist_ok=True)

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
print('준비 완료')

## 1. 표 확장 (5~10분)
- `food_table.csv` : 175종 × (알레르기 19항목 + 영양 6칸 + 레시피 수) → **이 파일은 Claude가 깃에 반영**
- `table_recipes.parquet` : 매칭된 레시피 원문 + 사진 주소 (재배포 금지 → 드라이브 `data1/`에만)

In [ ]:
RECIPES = f'{DATA}/table_recipes.parquet'
subprocess.run(['python', 'build_table.py', '--csv', CSV, '--out', f'{OUT}/food_table.csv', '--recipes-out', RECIPES],
               cwd=f'{REPO}/knowledge', check=True)

In [ ]:
import pandas as pd
t = pd.read_csv(f'{OUT}/food_table.csv')
print(t[['class', 'n_recipes', 'kcal', 'sodium_mg']].sort_values('n_recipes').head(15).to_string(index=False))

## 2. 사진 수집 (1시간 안팎)
- 음식마다 최대 `PER_CLASS`장 (음식 간 균형 + 시간 절약). 사진 주소가 같은 레시피는 한 번만.
- 짧은 변 256px JPEG로 줄여서 **음식 하나당 parquet 파일 하나**로 저장 (작은 파일 수만 개는 드라이브가 못 버팀).
- 이미 받은 음식은 건너뛰니까 끊기면 그냥 다시 실행.

In [ ]:
import io, hashlib, time
import requests
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

PER_CLASS = 400      # 늘리면 학습 데이터↑ 시간↑
WORKERS = 32
SIDE = 256

rec = pd.read_parquet(RECIPES, columns=['recipe_idx', 'label', 'title', 'image1'])
rec = rec[rec.image1.fillna('').str.startswith('http')].drop_duplicates(['label', 'image1'])
print(f'사진 주소 있는 (음식, 레시피) {len(rec):,}개, 음식 {rec.label.nunique()}종')

session = requests.Session()
session.headers['User-Agent'] = 'Mozilla/5.0 (course research project)'

def fetch(url):
    """→ (jpeg bytes 또는 None, 결과)"""
    try:
        r = session.get(url, timeout=8)
        if not r.ok:
            return None, f'http {r.status_code}'
        im = Image.open(io.BytesIO(r.content)).convert('RGB')
        if min(im.size) < 100:
            return None, 'too small'
        s = SIDE / min(im.size)
        if s < 1:
            im = im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC)
        buf = io.BytesIO(); im.save(buf, 'JPEG', quality=85)
        return buf.getvalue(), 'ok'
    except requests.RequestException as e:
        return None, type(e).__name__
    except Exception:
        return None, 'not image'

t0 = time.time()
for i, (label, g) in enumerate(rec.groupby('label')):
    path = f'{IMG}/{label}.parquet'
    if os.path.exists(path):
        continue
    g = g.sample(min(PER_CLASS, len(g)), random_state=0)
    with ThreadPoolExecutor(WORKERS) as ex:
        res = list(ex.map(fetch, g.image1))
    g = g.assign(jpg=[b for b, _ in res], why=[w for _, w in res])
    g[['label', 'image1', 'why']].to_csv(f'{IMG}/_log.csv', mode='a', header=not os.path.exists(f'{IMG}/_log.csv'), index=False)
    ok = g[g.jpg.notna()].drop(columns='why')
    ok = ok.assign(md5=ok.jpg.map(lambda b: hashlib.md5(b).hexdigest()))
    ok.to_parquet(path, index=False)
    print(f'[{i+1}] {label}: {len(ok)}/{len(g)} ({len(ok)/len(g):.0%})  {(time.time()-t0)/60:.0f}분')
print('끝')

## 3. 요약 · 가짜 사진 걸러내기
죽은 주소가 "사진 없음" 같은 똑같은 그림을 돌려주는 경우가 있어서, **같은 사진(md5)이 5번 넘게 나오면** 학습에서 빼고 센다.

In [ ]:
import glob
got = pd.concat([pd.read_parquet(p, columns=['label', 'md5']) for p in sorted(glob.glob(f'{IMG}/*.parquet'))])
fake = got.md5.value_counts().loc[lambda v: v > 5].index
pd.Series(list(fake), name='md5').to_csv(f'{IMG}/_fake_md5.csv', index=False)
print(f'가짜(반복) 사진 {len(fake)}종류, {got.md5.isin(fake).sum():,}장 → 학습 때 제외')

real = got[~got.md5.isin(fake)]
known = set(open(f'{REPO}/modeling/classes.txt').read().split())
s = pd.DataFrame({'시도': rec.groupby('label').size().clip(upper=PER_CLASS),
                  '받음': real.groupby('label').size()}).fillna(0).astype(int)
s['비율'] = (s['받음'] / s['시도']).round(2)
s['종류'] = ['Food-101' if c in known else '새 후보' for c in s.index]
s.sort_values('받음').to_csv(f'{OUT}/사진수집_요약.csv', encoding='utf-8-sig')
print(f"전체 {s['받음'].sum():,}장 / 시도 {s['시도'].sum():,} ({s['받음'].sum() / s['시도'].sum():.0%})")
print(s.groupby('종류')[['시도', '받음']].sum())
print('\n적게 받은 음식'); print(s.sort_values('받음').head(15).to_string())
print('\n실패 이유'); print(pd.read_csv(f'{IMG}/_log.csv').why.value_counts().head(8).to_string())

## 끝
다 돌리면 Claude에게 "03 다 돌렸어"라고 말하기. `찍먹_실험기록/표/`의 `food_table.csv`·`사진수집_요약.csv`를 읽어서
표를 깃에 반영하고, CLIP 검색기 · 처음 보는 음식 감지 학습 노트북을 만든다.